[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C17_Classical_NLP_Course/00_setup/00_environment_check.ipynb)

# 00 · 环境自检与方法论热身

本课全程 **纯 numpy、CPU 可跑**，**不调用任何 NLP 库**（NLTK/spaCy/gensim/sklearn 一律不用），用 numpy 把经典算法**从零实现**，再用 `assert` **对拍**绝对可信的参考。

这个 notebook 做三件事：① 确认环境；② 用最小例子体会经典 NLP 的两个基本操作（**数频率**与**算相似**）；③ 立下全课的纪律——**对拍（differential testing）** 与 **真实数据回退**。

## 1 · 环境自检

只需要 `numpy`。`matplotlib` 可选。**故意不导入任何 NLP 库**——本课的一切都从零写。

In [ ]:
import sys, platform
print('Python', sys.version.split()[0], '|', platform.system())
import numpy as np
print('numpy', np.__version__)
try:
    import matplotlib; print('matplotlib', matplotlib.__version__, '(可选)')
except Exception:
    print('matplotlib 未安装（可选，不影响课程）')
# 本课不依赖这些库；仅检测以提醒你：我们不会用它们
for lib in ['nltk', 'spacy', 'gensim', 'sklearn']:
    try:
        __import__(lib); print(f'(检测到 {lib}，但本课不使用它 —— 我们从零实现)')
    except Exception:
        print(f'(未安装 {lib}，完全没关系 —— 本课不需要)')
print('环境就绪 ✅')

## 2 · 经典 NLP 的第一个基本操作：数频率

几乎所有统计 NLP 都从**数数**开始：词频、共现、n-gram 计数。先用纯 Python 数一段文本的词频，建立「语料 → 计数 → 概率」的最小直觉。

In [ ]:
from collections import Counter
text = 'the cat sat on the mat the cat ran'
tokens = text.split()
counts = Counter(tokens)
N = len(tokens)
print('tokens (词例) :', tokens)
print('总 token 数 N =', N)
print('词频:', dict(counts))
# unigram 概率 = 词频 / 总数（最大似然估计 MLE）
probs = {w: c / N for w, c in counts.items()}
print('unigram 概率:', {w: round(pr, 3) for w, pr in probs.items()})
assert abs(sum(probs.values()) - 1.0) < 1e-9, '概率必须归一化为 1'
assert counts['the'] == 3 and counts['cat'] == 2
print('✅ 数频率 + 归一 = 最简单的语言模型（unigram）')

## 3 · 第二个基本操作：把词变成向量、算相似

另一条主线是**表示**：把词变成向量，用几何关系刻画语义。先看最朴素的 **one-hot**——它把任意两个不同词的相似度都视为 0（这正是模块 01 词嵌入要解决的问题）。

In [ ]:
vocab = sorted(set(tokens))
word2id = {w: i for i, w in enumerate(vocab)}
V = len(vocab)
print('词表:', vocab, '| V =', V)

def one_hot(w):
    v = np.zeros(V)
    v[word2id[w]] = 1.0
    return v

def cosine(u, v):
    denom = (np.linalg.norm(u) * np.linalg.norm(v))
    return float(u @ v / denom) if denom > 0 else 0.0

print('cos(cat, cat) =', cosine(one_hot('cat'), one_hot('cat')))
print('cos(cat, mat) =', cosine(one_hot('cat'), one_hot('mat')))
assert abs(cosine(one_hot('cat'), one_hot('cat')) - 1.0) < 1e-9
assert abs(cosine(one_hot('cat'), one_hot('mat'))) < 1e-9, 'one-hot 下不同词相似度恒为 0'
print('✅ one-hot 的致命缺陷：cat 和 mat 一样不相似（=0）—— 模块 01 用稠密向量解决它')

## 4 · 立纪律之一：对拍（differential testing）

本课每个算法都要和一个**绝对可信的参考**比对。最常用的参考是**暴力枚举**：小规模下，把所有可能性列出来直接算。

先热身：写一个「动态规划求和」与「暴力枚举求和」对拍。这正是模块 03 前向算法（DP 求所有路径概率和）验证方式的缩影。

In [ ]:
# 玩具问题：长度 T 的序列，每步可选 S 个状态，每条路径有个乘积权重
# 目标：所有路径权重之和。暴力 = 枚举 S^T 条路径；DP = 逐步累加（O(T*S^2)）
import itertools
rng = np.random.default_rng(0)
T, S = 4, 3
trans = rng.random((S, S))      # 相邻步之间的权重
init = rng.random(S)            # 第一步的权重

def brute_force_sum():
    total = 0.0
    for path in itertools.product(range(S), repeat=T):
        w = init[path[0]]
        for t in range(1, T):
            w *= trans[path[t-1], path[t]]
        total += w
    return total

def dp_sum():
    alpha = init.copy()                 # alpha[i] = 到当前步、停在状态 i 的权重和
    for t in range(1, T):
        alpha = alpha @ trans           # 一步转移 = 向量×矩阵（前向递推的本质）
    return alpha.sum()

bf, dp = brute_force_sum(), dp_sum()
print(f'暴力枚举 {S**T} 条路径之和 = {bf:.6f}')
print(f'动态规划 (O(T*S^2)) 之和 = {dp:.6f}')
assert np.allclose(bf, dp), 'DP 必须与暴力枚举逐位相等'
print('✅ 对拍通过：DP 用多项式时间算出了指数级求和 —— 这就是前向算法的灵魂（模块 03）')

## 5 · 立纪律之二：真实数据，联网失败回退内置

本课尽量用真实语料。但本环境可能无网络，所以每个真实数据胶囊都写成 `try: 下载 ... except: 回退到内置真实小样本`。

下面演示这个模式：尝试下载 **tiny-shakespeare**，失败则用一段内置的真实莎翁文本。无论哪条路径，后续算法处理完全一致。

In [ ]:
def load_text_with_fallback():
    '''尝试下载 tiny-shakespeare；失败回退到内置真实片段。返回 (text, source)。'''
    url = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
    try:
        import urllib.request
        with urllib.request.urlopen(url, timeout=5) as r:
            text = r.read().decode('utf-8')
        return text, 'downloaded (tiny-shakespeare)'
    except Exception:
        # 回退：内置真实莎翁片段（Hamlet / Sonnet，真实文本）
        text = ('To be, or not to be, that is the question: '
                'Whether tis nobler in the mind to suffer '
                'The slings and arrows of outrageous fortune, '
                'Or to take arms against a sea of troubles, '
                'And by opposing end them. ' * 3)
        return text, 'builtin fallback (real Shakespeare excerpt)'

text, source = load_text_with_fallback()
print('数据来源:', source)
print('文本长度:', len(text), '字符')
print('前 60 字符:', repr(text[:60]))
assert len(text) > 50, '无论下载还是回退，都应拿到非空文本'
print('✅ 真实数据回退模式就绪 —— 有网用全量，无网用内置真实样本，算法不变')

## 6 · 一个会贯穿全课的对拍工具

把「检查归一化」与「对拍」封装成小工具，后面每个模块都用它判定「我的实现 == 参考 / 概率合法」。

In [ ]:
def check_close(name, got, ref, atol=1e-9):
    '''对拍：实现结果 vs 参考。'''
    got_a, ref_a = np.asarray(got, float), np.asarray(ref, float)
    ok = np.allclose(got_a, ref_a, atol=atol)
    err = float(np.max(np.abs(got_a - ref_a))) if got_a.size else 0.0
    print(f'[{name:<28}] allclose={ok}  max|err|={err:.2e}')
    assert ok, f'{name} 与参考不一致！'

def check_is_distribution(name, probs, atol=1e-9):
    '''检查一组数构成合法概率分布（非负、和为 1）。'''
    arr = np.asarray(probs, float)
    assert (arr >= -atol).all(), f'{name}: 概率不能为负'
    s = float(arr.sum())
    print(f'[{name:<28}] sum={s:.6f}  min={arr.min():.4f}')
    assert abs(s - 1.0) < 1e-6, f'{name}: 概率和应为 1，实际 {s}'

check_close('DP == brute force', dp_sum(), brute_force_sum())
check_is_distribution('unigram probs', list(probs.values()))
print('\n✅ 全课统一裁判就绪：check_close（对拍）+ check_is_distribution（概率合法性）')

### 小结
- 经典 NLP 两大基本操作：**数频率**（→ 概率、语言模型）与 **算相似**（→ 表示、词向量）。
- one-hot 把不同词的相似度都当 0，无法表达语义 —— 这是模块 01 词嵌入的出发点。
- 全课纪律一：**纯 numpy 从零**，不调 NLP 库，每行代码都是理解。
- 全课纪律二：**对拍**，用暴力枚举/归一化检查把正确性钉成 `assert`。
- 全课纪律三：**真实数据 + 联网失败回退内置**，有网用全量、无网用真实小样本，算法路径不变。

下一站：**模块 01 · 词嵌入 word2vec** —— 把「词」变成有几何结构的稠密向量。